# 1. Get the sample and look at it

This notebook downloads 12 image sets from the ecdna-bench imaging resource
(BioImage Archive, accession S-BIAD4097) and shows what an image set contains:
the RGB FISH image, the manual region-of-interest (ROI) mask and the
gold-standard ecDNA mask. It then counts ecDNA in the gold standard the way the
benchmark does.

The 12 image sets are held-out test images, three per cell line: the lowest and
the highest gold-standard count, and one from the middle of the range (for
NCI-H2170 this is the example image used throughout the paper).

**Needs:** the installed `ecdna_bench` package and an internet connection.
Everything is written to `tutorial_data/` next to this notebook; notebooks 2
and 3 read from there. Downloads resume if interrupted: just run the cells again.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ecdna_bench

# All files of the tutorials live here (next to the notebook by default).
DATA = Path(os.environ.get("ECDNA_TUTORIAL_DATA", "tutorial_data")).expanduser().resolve()
print("ecdna_bench :", getattr(ecdna_bench, "__version__", "?"), "from", Path(ecdna_bench.__file__).parent)
print("data folder :", DATA)

SAMPLE = [
    ('COLO320DM', 'lowest', 'colo320dm_qpcr_jq1_24h_dmso_111'),
    ('COLO320DM', 'median', 'colo320dm_qpcr_jq1_24h_ctrl_39'),
    ('COLO320DM', 'highest', 'colo320dm_qpcr_jq1_24h_ctrl_114'),
    ('NCI-H2170', 'lowest', 'ncih2170_facs_fish_0223_low_her2_6'),
    ('NCI-H2170', 'example', 'ncih2170_facs_fish_0723_low_her2_52'),
    ('NCI-H2170', 'highest', 'ncih2170_antibiotics_ps_g_36'),
    ('NCI-H716', 'lowest', 'ncih716_jc_ctrl_2_30'),
    ('NCI-H716', 'median', 'ncih716_jc_alo_8nm_24h_13'),
    ('NCI-H716', 'highest', 'ncih716_jc_ctrl_1_66'),
    ('SNU16', 'lowest', 'snu16_jc_ctrl_45'),
    ('SNU16', 'median', 'snu16_jc_jq1_ic50_24h_67'),
    ('SNU16', 'highest', 'snu16_jc_ctrl_114'),
]

UIDS = [uid for _, _, uid in SAMPLE]
EXAMPLE_UID = "ncih2170_facs_fish_0723_low_her2_52"
DATA.mkdir(parents=True, exist_ok=True)

## Where the files come from

After the record is published, the files are public and nothing needs to be
entered. While the record is private (during peer review), the next cell asks
for the **share link** from the reviewer instructions
(`https://www.ebi.ac.uk/biostudies/bioimages/studies/S-BIAD4097?key=...`).
The link works like a password: the prompt hides it, and the notebook never
prints it or writes it to disk. You can also set it in the shell before
starting Jupyter: `export ECDNA_BIA_BASE_URL='<share link>'`.

In [ ]:
import getpass, json, re, shutil, time
import urllib.error, urllib.parse, urllib.request

ACCESSION = "S-BIAD4097"
INFO_API = f"https://www.ebi.ac.uk/biostudies/api/v1/studies/{ACCESSION}/info"
PUBLIC_FILES = f"https://ftp.ebi.ac.uk/pub/databases/biostudies/S-BIAD/097/{ACCESSION}/Files"
HEADERS = {"User-Agent": "ecdna-bench-tutorial/1.0"}

def _get(url, timeout=60):
    with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=timeout) as r:
        return r.read()

def hide_key(url):
    """The address with any access key replaced by <key>."""
    url = re.sub(r"(/\.private/\d+/)[^/]+", r"\1<key>", url)
    return re.sub(r"([?&](?:key|accessKey|token)=)[^&#/]+", r"\1<key>", url, flags=re.I)

def files_location(link=""):
    """URL of the study's Files folder, from a share link, a Files URL, or nothing (public)."""
    link = (link or "").strip()
    if link and not link.startswith(("http://", "https://")):
        raise ValueError("That is not a web address; paste the whole share link (it starts with https://).")
    key = ""
    if link:
        parts = urllib.parse.urlsplit(link)
        if "/studies/" in parts.path:
            key = urllib.parse.parse_qs(parts.query).get("key", [""])[0]
        if not key:
            return link.rstrip("/")            # already a Files address
    info_url = f"{INFO_API}?key={urllib.parse.quote(key)}" if key else INFO_API
    try:
        info = json.loads(_get(info_url, timeout=30).decode("utf-8"))
        loc = info.get("httpLink") or info.get("ftpHttp_link")
    except urllib.error.HTTPError as exc:
        if key:
            raise RuntimeError(f"The BioStudies API refused the share link (HTTP {exc.code}). "
                               "Check that the whole link was copied.") from None
        loc = None
    except (urllib.error.URLError, OSError, ValueError):
        if key:
            raise RuntimeError("Could not reach the BioStudies API to resolve the share link.") from None
        loc = None
    if not loc:
        if key:
            raise RuntimeError("The BioStudies API returned no file location for this share link.")
        return PUBLIC_FILES
    loc = str(loc).rstrip("/")
    return loc if loc.endswith("/Files") else loc + "/Files"

def download(rel, retries=3):
    """Download one archive file to DATA/<rel> (kept if already complete)."""
    dest = DATA / rel
    if dest.is_file() and dest.stat().st_size > 0:
        return "kept"
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(dest.name + ".part")
    url = FILES + "/" + urllib.parse.quote(rel)
    last = None
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=120) as r, \
                 open(part, "wb") as fh:
                expected = r.headers.get("Content-Length")
                shutil.copyfileobj(r, fh, 1 << 20)
            if expected is not None and part.stat().st_size != int(expected):
                raise IOError("incomplete download")
            part.replace(dest)
            return "downloaded"
        except (urllib.error.URLError, OSError) as exc:
            last = exc
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"could not download {rel} ({last.__class__.__name__}: "
                       f"{getattr(last, 'code', '') or getattr(last, 'reason', '')})")

In [ ]:
link = os.environ.get("ECDNA_BIA_BASE_URL", "").strip()
if not link:
    try:
        link = getpass.getpass("Share link (press Enter if the record is public): ").strip()
    except Exception:          # no interactive input available (e.g. a headless run)
        link = ""
FILES = files_location(link)
del link
print("files from:", hide_key(FILES))

## Select the 12 image sets

The archive publishes one file list per section. The next cell reads four of
them (images, gold standard, ROI, predictions) and picks the files of the 12
image sets: the RGB image, the gold-standard mask, the manual ROI mask, and the
masks predicted by the six benchmarked methods (used in notebook 3).

In [ ]:
LISTS = ["filelist_images.tsv", "filelist_gt.tsv", "filelist_roi.tsv", "filelist_predictions.tsv"]
PRED_FOLDERS = {                        # archive folder -> method name used in the paper
    "eccount_peaks": "ecCount (peaks)",
    "eccount_threshold": "ecCount (threshold mask)",
    "label_engine": "Label Engine",
    "mia": "MIA",
    "classical_optimised": "Classic (after opt)",
    "ecseg": "ecSeg",
}

tables = {}
for name in LISTS:
    dest = DATA / "filelists" / name
    if not (dest.is_file() and dest.stat().st_size > 0):
        try:
            data = _get(FILES + "/" + name)
        except urllib.error.HTTPError as exc:
            raise RuntimeError(f"Could not read {name} (HTTP {exc.code}). If the record is not public "
                               "yet, run the previous cell again and paste the share link.") from None
        except (urllib.error.URLError, OSError) as exc:
            raise RuntimeError(f"Could not reach the archive to read {name} "
                               f"({getattr(exc, 'reason', exc.__class__.__name__)}). "
                               "Check the internet connection and run this cell again.") from None
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
    t = pd.read_csv(dest, sep="\t", dtype=str)
    t["uid"] = t["Files"].map(lambda p: Path(p).stem)
    t["folder"] = t["Files"].map(lambda p: "/".join(p.split("/")[:-1]))
    tables[name] = t[t["uid"].isin(UIDS)]

images = tables["filelist_images.tsv"]
rgb = images[images["folder"] == "images/rgb"].set_index("uid")
gt = tables["filelist_gt.tsv"].query("folder == 'images/gt_image'").set_index("uid")
roi = tables["filelist_roi.tsv"].query("folder == 'images/roi_mask'").set_index("uid")
preds = tables["filelist_predictions.tsv"].copy()
preds["method_folder"] = preds["folder"].str.split("/").str[1]
preds = preds[preds["method_folder"].isin(PRED_FOLDERS)]

wanted = list(rgb.loc[UIDS, "Files"]) + list(gt.loc[UIDS, "Files"]) \
       + list(roi.loc[UIDS, "Files"]) + list(preds["Files"])
missing = [u for u in UIDS if u not in rgb.index or u not in gt.index or u not in roi.index]
assert not missing, f"not in the file lists: {missing}"
assert len(preds) == len(UIDS) * len(PRED_FOLDERS), f"expected {len(UIDS) * len(PRED_FOLDERS)} prediction files, found {len(preds)}"
print(f"{len(wanted)} files to fetch for {len(UIDS)} image sets")

In [ ]:
t0, status = time.time(), {"downloaded": 0, "kept": 0}
for i, rel in enumerate(wanted, 1):
    status[download(rel)] += 1
    if i % 20 == 0 or i == len(wanted):
        print(f"  {i}/{len(wanted)} files", flush=True)
size_mb = sum((DATA / rel).stat().st_size for rel in wanted) / 1e6
print(f"done in {time.time() - t0:.0f} s: {status['downloaded']} downloaded, {status['kept']} already here, "
      f"{size_mb:.0f} MB in {DATA}")

sample = pd.DataFrame([{
    "uid": uid, "cell_line": cl, "slot": slot,
    "split": rgb.loc[uid, "Split"],
    "gt_count_archive": int(rgb.loc[uid, "ecDNA Count"]),
    "rgb": rgb.loc[uid, "Files"], "gt": gt.loc[uid, "Files"], "roi": roi.loc[uid, "Files"],
} for cl, slot, uid in SAMPLE])
sample.to_csv(DATA / "sample.csv", index=False)
preds.assign(method=preds["method_folder"].map(PRED_FOLDERS))[["uid", "method", "Files"]] \
     .rename(columns={"Files": "path"}).to_csv(DATA / "predictions.csv", index=False)
sample[["cell_line", "slot", "uid", "split", "gt_count_archive"]]

## One image set

The RGB image is a FISH image of a metaphase spread: DNA (DAPI) in blue and the
amplified locus in the FISH colour. The manual ROI mask marks the metaphase
spread; everything outside it is ignored. The gold-standard mask marks each
ecDNA as a small object. Below: the paper's example image, with the ROI outline
and a zoomed region.

In [ ]:
import cv2
cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)   # hide notes about extra TIFF tags

def read_rgb(path):
    """RGB image as uint8 (H, W, 3), read the same way as the benchmark."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"cannot read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def read_gray(path):
    """Single-channel image (TIFF or PNG); colour images are reduced by their maximum."""
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        import tifffile
        img = tifffile.imread(str(path))
    img = np.asarray(img)
    return img.max(axis=2) if img.ndim == 3 else img

def load_sample():
    """The table written by notebook 1."""
    table = DATA / "sample.csv"
    if not table.is_file():
        raise FileNotFoundError(f"{table} not found. Run notebook 1 first "
                                "(or set ECDNA_TUTORIAL_DATA to its data folder).")
    return pd.read_csv(table)

row = load_sample().set_index("uid").loc[EXAMPLE_UID]
img, roi_mask, gt_mask = read_rgb(DATA / row["rgb"]), read_gray(DATA / row["roi"]), read_gray(DATA / row["gt"])
print("image", img.shape, img.dtype, "| ROI", roi_mask.shape, "| gold standard", gt_mask.shape, np.unique(gt_mask))

x0, y0, x1, y1 = 1247, 1225, 1458, 1353        # zoom window used in the paper's figures
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img); ax[0].contour(roi_mask > 0, levels=[0.5], colors="w", linewidths=1)
ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=1.5))
ax[0].set_title("RGB image, ROI outline (white), zoom (yellow)")
ax[1].imshow(img[y0:y1, x0:x1]); ax[1].set_title("zoom")
ax[2].imshow(img[y0:y1, x0:x1])
ax[2].contour(gt_mask[y0:y1, x0:x1] > 0, levels=[0.5], colors="lime", linewidths=1)
ax[2].set_title("zoom with gold-standard objects (green)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Count ecDNA the way the benchmark does

The benchmark counts objects as 8-connected components of at least 3 px in a
mask. The same rule is applied to every method's prediction and to the gold
standard, so counts are comparable. `ecdna_bench.evaluation.objects_from_mask`
implements it; the counts below should equal the count recorded in the archive.

In [ ]:
from ecdna_bench.evaluation import objects_from_mask

sample = load_sample()
sample["gt_count"] = [len(objects_from_mask(read_gray(DATA / p), min_area=3, connectivity=8,
                                            attach_mask=False)) for p in sample["gt"]]
sample["matches_archive"] = sample["gt_count"] == sample["gt_count_archive"]
print(f"counts equal to the archive for {int(sample['matches_archive'].sum())} of {len(sample)} image sets")
sample[["cell_line", "slot", "gt_count", "gt_count_archive", "matches_archive"]]

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 15))
for ax, r in zip(axes.ravel(), sample.itertuples()):
    im, m = read_rgb(DATA / r.rgb), read_gray(DATA / r.roi) > 0
    ys, xs = np.where(m)
    ax.imshow(im[ys.min():ys.max() + 1, xs.min():xs.max() + 1])
    ax.set_title(f"{r.cell_line}, {r.slot}: {r.gt_count} ecDNA", fontsize=10)
    ax.axis("off")
plt.suptitle("The 12 image sets, cropped to the ROI", y=1.0)
plt.tight_layout(); plt.show()

## Next

* **Notebook 2** runs ecCount on these 12 images on the CPU.
* **Notebook 3** scores ecCount and the other benchmarked methods against the
  gold standard on the same images.

To download more (a whole split, one cell line, or the full resource), use
`scripts/fetch_bia_subset.py` from the repository; see `docs/TUTORIAL_EXTERNAL.md`.